# 2. Running a SWIFT zoom

Running a zoom with SWIFT couldn't really be easier... all we need is to include the `ZoomRegion` block in the parameter file and run with the ``--zoom`` runtime flag.

The parameters in the ``ZoomRegion`` control the geometry of the zoom cells (the SWIFT cells that cover the zoom region). Even more so then uniform box simulations these choices have a significant impact on the performance of the simulation. 

These parameters include (but some more esoteric ones are not shown here):

```yaml
ZoomRegion:
  region_pad_factor:          1.1    # Padding for the high res particle distribution in the ICs?
  zoom_top_level_depth:       2      # The depth of the zoom top level cells in a single background cell.
  zoom_min_top_level_cells:   -1     # (Optional) Minimum target zoom cdim. If positive, overrides zoom_top_level_depth cells 
  max_com_dx:                 0.1    # (Optional) The maximum allowed displacement of the centre of mass of the zoom region from the centre of the box in units of the zoom region extent. When this is exceeded, we rebuild the zoom region.
```

## Build SWIFT

Like with MUSIC-2, we need to build SWIFT either locally or on your HPC. This carries all the same caveats as before, and once again, if you read the README you should have already run the build and can proceed to the next section.

**Local machine**

```bash
scripts/build_swift.sh
```

**COSMA8 login node**

```bash
module purge
module load intel_comp/2024.2.0 compiler-rt tbb compiler mpi ucx/1.17.0 parallel_hdf5/1.14.4 fftw/3.3.10 gsl/2.8
JOBS=8 scripts/build_swift.sh
```

## Choose the geometry

We will here focus on chosing three parameters that strongly affect the performance of the simulation:

- **Zoom cells per axis:** how finely SWIFT divides the high-resolution region.
- **Background cells per axis:** the grid covering the full box.
- **PM mesh resolution:** the Fourier mesh used for PM gravity.

Sadly there is no obvious golden bullet choice, as the optimal values depend strongly on the zoom region set up. The three values appear directly in the commands below and ~~we're~~ you're going to have a bit of an experiment. 

The first step in this experiment is to visualise the zoom region and the zoom cells. Thankfully, we have machinery to do this easily. We can pass `--dump-zoom-geometry` at runtime and SWIFT will do a dry run where it loads the particles, constructs the geometry, writes a set of diagnostic files, and stops before taking a simulation step. We can then use the `plot_zoom_geometry.py` script from the tools directory to visualise the geometry.

To run this, do either of the following, depending on whether you are running locally or on COSMA8/a HPC:

**Local machine**

```bash
scripts/run_swift.sh geometry 4 \
  ZoomRegion:zoom_min_top_level_cells:8 \
  ZoomRegion:bkg_top_level_cells:33 \
  Gravity:mesh_side_length:256
```

**COSMA8**

```bash
sbatch --export=ALL,MODE=geometry,ZOOM_MIN_CELLS=8,BACKGROUND_CELLS=33,MESH_SIDE_LENGTH=256 submission_scripts/swift_cosma8.slurm
```

Again, if running remotely, pull the results back to your local machine for plotting:

```bash
rsync -avz -P <user>@<hpcdomain>:<your path to repo>/runs/zoom/ runs/zoom/
```

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plot_dir = Path("../runs/zoom/geometry_plots")
for path in sorted(plot_dir.glob("*.png")):
    print(path.name)
    display(Image(filename=path))

## Run the simulation

Now we have a good idea of what the geometry looks like, we can run the simulation. Again, this is done exactly like running SWIFT for a uniform box, but with the ``--zoom`` flag. Here we are just using a simple script to run swift. Please do look at it's contents to see what it is doing, and feel free to edit it to suit your needs. 

**Local machine**

```bash
scripts/run_swift.sh run 4 16 \
  ZoomRegion:zoom_min_top_level_cells:8 \
  ZoomRegion:bkg_top_level_cells:33 \
  Gravity:mesh_side_length:256
```

**COSMA8**

```bash
sbatch --export=ALL,STEPS=16,ZOOM_MIN_CELLS=8,BACKGROUND_CELLS=33,MESH_SIDE_LENGTH=256 submission_scripts/swift_cosma8.slurm
```

For a simulation run on an HPC, only pull the verbose log back to your local machine:

```bash
rsync -avz -P <user>@<hpcdomain>:<your path to repo>/runs/zoom/output.log runs/zoom/
```

Once the run is finished, you have completed the workshop and can either explore different zoom geometries to get a feel for things or try running with full physics to see how that changes things.

## Optional: SWIFT-EAGLE

The main exercise is DMO. If there is time, and you want to prove to yourself it works, you can try running with full physics. To do so you will need to reconfigure and download some input files. See the commands below to help.

**Local machine**

```bash
scripts/build_swift_eagle.sh
scripts/download_eagle_tables.sh
```

**COSMA8 login node**

```bash
module purge
module load intel_comp/2024.2.0 compiler-rt tbb compiler mpi ucx/1.17.0 parallel_hdf5/1.14.4 fftw/3.3.10 gsl/2.8
JOBS=8 scripts/build_swift_eagle.sh
scripts/download_eagle_tables.sh
```